# 04 Small CNN trained from scratch

Functions: `src/piece_of_crop/cnn.py` and `src/piece_of_crop/train.py`.

**Purpose.** Train a small convolutional network from scratch on RGB (3 channels) and on all 13 bands, with the same split, settings and seeds, and show the training and validation curves.

**Reads.** The split file and band statistics from notebook 02.
**Saves.** Test predictions as `results/predictions/cnn_<input>_seed<n>.csv` and the training curves in `results/figures/`. Model weights stay local and are not committed.

**Why.** The architecture follows the course CNN pattern of convolution, batch normalisation, ReLU and pooling blocks with global average pooling (lecture_8_DL_for_CV_1_compressed.pdf, slide 40; CNN_Class_Competition_2_Food_Classification_Explained.pdf, section 2). A convolution sums over all input channels (lecture_8, slide 32), so the same network takes 3 or 13 channels by changing only `in_channels`.

## Setup: shared data for every model

This cell is the same in notebooks 03, 04 and 05, so every model starts from exactly the same data. It reads the split (`results/split.csv`) and the normalisation statistics (`results/band_stats.csv`) saved by notebook 02, and opens the cache of all 27,000 patches. The first notebook you run on your computer builds the cache once (`data/eurosat_cache.npy`, about 2.9 GB, not in Git); after that this cell only loads it. Do not change this cell before discussing.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from piece_of_crop import get_cache, make_loaders, read_split

SEED = 0
project_root = Path.cwd().parents[0]
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
print("Patches per split:", split["split"].value_counts().to_dict())

## 1. Settings

same settings for rgb and 13 bands, only the input channels change. adam, lr 1e-3, batch 128, max 30 epochs, stops early if val loss doesnt improve for 5 epochs. 3 seeds each.

training takes a while on a laptop so lower `MAX_EPOCHS` if you just want to test it

In [ ]:
import matplotlib.pyplot as plt

from piece_of_crop.cnn import build_cnn, count_parameters
from piece_of_crop.train import get_device, run_experiment

SEEDS = [0, 1, 2]
INPUTS = ["rgb", "13band"]
MAX_EPOCHS = 30
PATIENCE = 5
LR = 1e-3

device = get_device()
print("Device:", device)
for input_type in INPUTS:
    print(input_type, "parameters:", count_parameters(build_cnn(input_type)))

## 2. Train every input and seed

predictions go to `results/predictions/cnn_<input>_seed<n>.csv`, curves to `results/training/` + `results/figures/`, weights to `models/` (not in git).

scores get saved after every run so if it crashes halfway just rerun the cell, it skips what's done

In [ ]:
results_path = results_dir / "cnn_results.csv"
done = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()

for input_type in INPUTS:
    for seed in SEEDS:
        if len(done) and ((done["input"] == input_type) & (done["seed"] == seed)).any():
            print(f"cnn_{input_type}_seed{seed} already done, skipping")
            continue
        row = run_experiment("cnn", build_cnn, split, cache, band_stats, input_type, seed,
                             project_root, max_epochs=MAX_EPOCHS, lr=LR, patience=PATIENCE,
                             device=device)
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(results_path, index=False)

done

## 3. Results: RGB vs 13 bands

mean and std over the 3 seeds, this goes into the big comparison table in 06

In [ ]:
results = pd.read_csv(results_dir / "cnn_results.csv")
summary = (results.groupby("input")[["val_accuracy", "val_macro_f1", "test_accuracy",
                                      "test_macro_f1", "best_epoch", "train_time_s"]]
           .agg(["mean", "std"]).round(3).reindex(INPUTS))
summary

## 4. Training curves

seed 0, both inputs. dashed = validation. if val loss goes up while train loss keeps dropping its overfitting (thats why we have early stopping)

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
for input_type, colour in zip(INPUTS, ["tab:blue", "tab:orange"]):
    history = pd.read_csv(results_dir / "training" / f"cnn_{input_type}_seed0.csv")
    for ax, metric in [(ax_loss, "loss"), (ax_acc, "acc")]:
        ax.plot(history["epoch"], history[f"train_{metric}"], color=colour, label=f"{input_type} train")
        ax.plot(history["epoch"], history[f"val_{metric}"], color=colour, linestyle="--",
                label=f"{input_type} val")
ax_loss.set(title="Loss", xlabel="epoch")
ax_acc.set(title="Accuracy", xlabel="epoch")
for ax in (ax_loss, ax_acc):
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle("Small CNN, seed 0")
fig.tight_layout()
fig.savefig(figures_dir / "cnn_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. notes for report

- architecture: 3 conv blocks (32/64/128 filters), global avg pooling, dropout 0.3, 1 linear layer. params rgb vs 13 band: ...
- why small: images are only 64x64 and ~19k training images so a big network would overfit. batchnorm, dropout and the flips/rotations help too
- test acc / macro f1, rgb vs 13 band (mean ± std): ...
- best epoch + training time: ...
- what the curves show: ...